In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

df = spark.table("silver.combined_taxi_trips")

df_model = (
    df.select(
        "pulocationid",
        "dolocationid",
        "service_type",
        "pickup_datetime",
        "trip_time",
        "total_amount"
    )
    .dropna()
    .filter(
        (F.col("service_type") != "Unknown") &
        (F.col("pickup_datetime") > F.lit("2022-12-12"))
    )
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("time_group", F.floor(F.col("pickup_hour") / 4))
)

df_model = df_model.sampleBy(
    "service_type",
    fractions={
        "Green": 1.0,      # keep all, only ~2M anyway
        "Uber": 0.01,    # ~2.0M
        "Lyft": 0.03,    # ~2.0M
        "Yellow": 0.03   # ~2.0M
    },
    seed=42
)

indexer = StringIndexer(
    inputCol="service_type",
    outputCol="service_type_idx",
    handleInvalid="keep"
)

indexer_model = indexer.fit(df_model)
df_model = indexer_model.transform(df_model)

assembler = VectorAssembler(
    inputCols=["pulocationid", "dolocationid", "service_type_idx", "time_group"],
    outputCol="features"
)

df_model = assembler.transform(df_model)

train, test = df_model.randomSplit([0.8, 0.2], seed=42)

model_time = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_time"
).fit(train)

model_cost = RandomForestRegressor(
    featuresCol="features",
    labelCol="total_amount"
).fit(train)

base_path = "dbfs:/FileStore/taxi_models"

indexer_model.write().overwrite().save(f"{base_path}/service_type_indexer")
model_time.write().overwrite().save(f"{base_path}/trip_time_model")
model_cost.write().overwrite().save(f"{base_path}/total_amount_model")